# Example 11 - PnL SWIG API Showcase

This notebook demonstrates the currently available PnL-related SWIG APIs using
small in-memory objects so the flow runs quickly and deterministically.

Scope in this example:
- Sensitivity stream iteration and RiskFilter usage
- Historical scenario generation
- Sensitivity-based PnL and covariance computation
- PnlAnalytic / PnlExplainAnalytic wrapper construction and downcasting

Category aggregation via SensitivityAggregator is not included in this notebook.

## 1 - Environment setup

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

base_dir = Path.cwd()
output_dir = base_dir / "Output"
output_dir.mkdir(parents=True, exist_ok=True)

oreswig_pkg = os.environ.get("ORESWIG_PKG", "")
if oreswig_pkg:
    for p in reversed([x for x in oreswig_pkg.split(";") if x]):
        if p not in sys.path:
            sys.path.insert(0, p)

from ORE import *
import ORE

print("Loaded ORE from:", ORE.__file__)
print("Has PNLCalculator:", hasattr(ORE, "PNLCalculator"))
print("Has HistoricalSensiPnlCalculator:", hasattr(ORE, "HistoricalSensiPnlCalculator"))

Loaded ORE from: C:\Dev\oreplus\build\ore\ORE-SWIG\ORE.py
Has PNLCalculator: True
Has HistoricalSensiPnlCalculator: True


## 2 - Build small in-memory sensitivities and scenarios

In [2]:
factory = ORE.SimpleScenarioFactory()

ir_key = ORE.RiskFactorKey(ORE.RiskFactorKey.KeyType_DiscountCurve, "EUR", 0)
fx_key = ORE.RiskFactorKey(ORE.RiskFactorKey.KeyType_FXSpot, "EURUSD", 0)

d1 = ORE.Date(3, ORE.January, 2020)
d2 = ORE.Date(6, ORE.January, 2020)
d3 = ORE.Date(7, ORE.January, 2020)

s1 = factory.buildScenario(d1, True)
s2 = factory.buildScenario(d2, True)
s3 = factory.buildScenario(d3, True)

for s, ir, fx in [(s1, 0.9950, 1.1200), (s2, 0.9949, 1.1210), (s3, 0.9948, 1.1220)]:
    s.add(ir_key, ir)
    s.add(fx_key, fx)

scenario_vec = ORE.ScenarioVector()
for s in (s1, s2, s3):
    scenario_vec.append(s)

date_set = ORE.DateSet()
for d in (d1, d2, d3):
    date_set.insert(d)

loader = ORE.HistoricalScenarioLoader(scenario_vec, date_set)
ret_cfg = ORE.ReturnConfiguration()
hist_gen = ORE.HistoricalScenarioGenerator(loader, factory, ret_cfg)

stream = ORE.SensitivityInMemoryStream()
stream.add(ORE.SensitivityRecord("swap_1", False, ir_key, "EUR discount", 0.0001,
                                 ORE.RiskFactorKey(), "", 0.0, "EUR", 100.0, 5000.0, 0.0))
stream.add(ORE.SensitivityRecord("fxfwd_1", False, fx_key, "EURUSD spot", 0.0001,
                                 ORE.RiskFactorKey(), "", 0.0, "EUR", 80.0, -3000.0, 0.0))

print("Scenario pairs:", hist_gen.numScenarios())
print("Sensitivity records:", len(stream.readAll()))

Scenario pairs: 2
Sensitivity records: 2


## 3 - Iterate sensitivity stream and inspect records

In [3]:
rows = []
for rec in stream:
    rows.append({
        "tradeId": rec.tradeId,
        "key": rec.desc_1,
        "delta": rec.delta,
        "gamma": rec.gamma,
    })

sens_df = pd.DataFrame(rows)
sens_df

,tradeId,key,delta,gamma
0,swap_1,EUR discount,5000.0,0.0
1,fxfwd_1,EURUSD spot,-3000.0,0.0


## 4 - Split records by market risk class using RiskFilter

In [4]:
rf_ir = ORE.RiskFilter(ORE.MarketRiskConfiguration.RiskClass_InterestRate,
                       ORE.MarketRiskConfiguration.RiskType_All)
rf_fx = ORE.RiskFilter(ORE.MarketRiskConfiguration.RiskClass_FX,
                       ORE.MarketRiskConfiguration.RiskType_All)

records = stream.readAll()
ir_count = sum(1 for r in records if rf_ir.allow(r.key_1))
fx_count = sum(1 for r in records if rf_fx.allow(r.key_1))

pd.DataFrame([
    {"riskClass": "InterestRate", "count": ir_count},
    {"riskClass": "FX", "count": fx_count},
])

,riskClass,count
0,InterestRate,1
1,FX,1


## 5 - Historical scenario generator overview

In [5]:
pd.DataFrame({
    "startDate": [d.ISO() for d in hist_gen.startDates()],
    "endDate": [d.ISO() for d in hist_gen.endDates()],
})

,startDate,endDate
0,2020-01-03,2020-01-06
1,2020-01-06,2020-01-07


## 6 - Compute sensitivity-based PnL and covariance

In [6]:
base = factory.buildScenario(d1, True)
base.add(ir_key, 0.9950)
base.add(fx_key, 1.1200)
hist_gen.setBaseScenario(base)

num_scenarios = hist_gen.numScenarios()
ids = ORE.StringSet()
ids.insert("DiscountCurve/EUR/0")
ids.insert("FXSpot/EURUSD/0")

shift_cube = ORE.DoublePrecisionInMemoryCubeN(d1, ids, [d1], num_scenarios)
for i in range(num_scenarios):
    shift_cube.set(0.0001 * (i + 1), 0, 0, i, 0)
    shift_cube.set(0.0010 * (i + 1), 1, 0, i, 0)

rf_keys = ORE.RiskFactorKeyVector()
rf_keys.append(ir_key)
rf_keys.append(fx_key)

record_set = ORE.SensitivityRecordSet()
for rec in stream.readAll():
    record_set.insert(rec)

period = ORE.TimePeriod([d1, d3])
pnl_calc = ORE.PNLCalculator(period)
pnl_calcs = ORE.PNLCalculatorVector()
pnl_calcs.append(pnl_calc)

cov_calc = ORE.CovarianceCalculator(period)
keys_for_cov = ORE.RiskFactorKeySizePairSet()
keys_for_cov.insert(ir_key, 0)
keys_for_cov.insert(fx_key, 1)
cov_calc.initialise(keys_for_cov)

sensi_pnl = ORE.HistoricalSensiPnlCalculator(hist_gen, stream)
sensi_pnl.calculateSensiPnl(record_set, rf_keys, shift_cube, pnl_calcs, cov_calc)
cov_calc.populateCovariance(keys_for_cov)

pnl_df = pd.DataFrame({
    "scenario": list(range(1, len(pnl_calc.pnls()) + 1)),
    "all_pnl": list(pnl_calc.pnls()),
    "fo_pnl": list(pnl_calc.foPnls()),
})
pnl_df.to_csv(output_dir / "pnl_summary.csv", index=False)

print("Covariance matrix:")
for i in range(cov_calc.covariance().rows()):
    print([cov_calc.covariance()[i][j] for j in range(cov_calc.covariance().columns())])

pnl_df

Covariance matrix:
[2.4999999999999996e-09, 2.5e-08]
[2.5e-08, 2.5e-07]


,scenario,all_pnl,fo_pnl
0,1,-2.5,-2.5
1,2,-5.0,-5.0


## 7 - Reuse a detailed PnL input set and check analytic wrappers

In [7]:
example62 = (base_dir.parent / "Example_8" / "Input" / "Example_62").resolve()

def read_xml(name):
    return (example62 / name).read_text(encoding="utf-8")

inputs = ORE.InputParameters()
inputs.setAsOfDate("2023-01-31")
inputs.setResultsPath(str(output_dir))
inputs.setAllFixings(True)
inputs.setEntireMarket(True)
inputs.setBaseCurrency("USD")
inputs.setCurveConfigs(read_xml("curveconfig.xml"))
inputs.setConventions(read_xml("conventions.xml"))
inputs.setPricingEngine(read_xml("pricingengine.xml"))
inputs.setTodaysMarketParams(read_xml("todaysmarket.xml"))
inputs.setPortfolio(read_xml("portfolio.xml"))
inputs.insertAnalytic("PNL")
inputs.insertAnalytic("PNL_EXPLAIN")
inputs.setScenarioSimMarketParamsFromFile(str(example62 / "simulation.xml"))
inputs.setSensiSimMarketParams(read_xml("simulation.xml"))
inputs.setSensiScenarioData(read_xml("sensitivity.xml"))

pnl_analytic = ORE.PnlAnalytic()
pnl_explain_analytic = ORE.PnlExplainAnalytic()

print("Example_62 input path:", example62)
print("PnlAnalytic constructible:", pnl_analytic is not None)
print("PnlExplainAnalytic constructible:", pnl_explain_analytic is not None)
print("asPnlAnalytic helper exported:", hasattr(ORE, "asPnlAnalytic"))
print("asPnlExplainAnalytic helper exported:", hasattr(ORE, "asPnlExplainAnalytic"))

Example_62 input path: C:\Dev\oreplus\ore\Examples\ORE-Python\Notebooks\Example_8\Input\Example_62
PnlAnalytic constructible: True
PnlExplainAnalytic constructible: True
asPnlAnalytic helper exported: True
asPnlExplainAnalytic helper exported: True


## 8 - Summary

In [8]:
summary_df = pd.DataFrame([
    {"metric": "num_scenarios", "value": hist_gen.numScenarios()},
    {"metric": "num_sensitivity_records", "value": len(stream.readAll())},
    {"metric": "num_pnl_points", "value": len(pnl_calc.pnls())},
])
summary_df.to_csv(output_dir / "summary.csv", index=False)
summary_df

,metric,value
0,num_scenarios,2
1,num_sensitivity_records,2
2,num_pnl_points,2
